## 0. Подготовка

In [ ]:
import os, re, json, time, random, math, urllib.parse
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import requests
from sklearn.feature_extraction.text import TfidfVectorizer
from rank_bm25 import BM25Okapi

from catboost import CatBoostRanker, Pool
import lightgbm as lgb

import ir_measures
from ir_measures import nDCG, MAP, P, read_trec_qrels

RNG_SEED = 42
np.random.seed(RNG_SEED); random.seed(RNG_SEED)

WIKI_DIR   = 'wikIR1k'
DOCS_CSV   = 'documents.csv'
IMAT_DIR   = 'imat2009_new_split'
IMAT_OUT   = 'imat2009_letor'
MIRAGE_DIR = 'mirage'
os.makedirs(IMAT_OUT, exist_ok=True)

def ndcg_at_k(y_true, y_pred, groups, k=10):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred); groups = np.asarray(groups)
    out = []
    i = 0
    while i < len(groups):
        j = i
        while j < len(groups) and groups[j] == groups[i]:
            j += 1
        yt = y_true[i:j]; yp = y_pred[i:j]
        order = np.argsort(-yp)
        sorted_y = yt[order][:k]
        gains = 2.0**sorted_y - 1
        discounts = 1.0/np.log2(np.arange(2, len(sorted_y)+2))
        dcg = (gains * discounts).sum()
        ideal = 2.0**np.sort(yt)[::-1][:k] - 1
        idcg = (ideal * discounts[:len(ideal)]).sum()
        out.append(dcg/idcg if idcg > 0 else 1.0)
        i = j
    return float(np.mean(out)) if out else 0.0

/Users/smakov/.pyenv/versions/3.11.9/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. CatBoost MSLR-WEB10k

In [2]:
from catboost.datasets import msrank_10k
train_df, test_df = msrank_10k()
print('train:', train_df.shape, ' test:', test_df.shape)

def to_pool(df):
    df = df.sort_values(1).reset_index(drop=True)  # группы должны идти подряд
    y = df[0].values
    qid = df[1].astype(int).values
    X = df.drop(columns=[0, 1]).values
    return Pool(X, label=y, group_id=qid)

train_pool = to_pool(train_df)
test_pool  = to_pool(test_df)
print('n_queries train:', len(set(train_df[1])), ' test:', len(set(test_df[1])))

train: (10000, 138)  test: (10000, 138)
n_queries train: 87  test: 88


In [3]:
model_msrank = CatBoostRanker(
    loss_function='YetiRank',
    iterations=500,
    learning_rate=0.05,
    depth=6,
    random_seed=RNG_SEED,
    eval_metric='NDCG:top=10;type=Exp',
    early_stopping_rounds=50,
    verbose=100,
)
model_msrank.fit(train_pool, eval_set=test_pool, use_best_model=True)

pred_msrank = model_msrank.predict(test_pool)
y_te_m = test_df.sort_values(1)[0].values
q_te_m = test_df.sort_values(1)[1].astype(int).values
ndcg10_msrank = ndcg_at_k(y_te_m, pred_msrank, q_te_m, k=10)
print(f'NDCG@10 (test, MSLR-WEB10k): {ndcg10_msrank:.4f}')

0:	test: 0.2681861	best: 0.2681861 (0)	total: 72ms	remaining: 35.9s


100:	test: 0.4233644	best: 0.4233644 (100)	total: 920ms	remaining: 3.63s


200:	test: 0.4354599	best: 0.4387981 (195)	total: 1.72s	remaining: 2.56s


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.4439364597
bestIteration = 213

Shrink model to first 214 iterations.
NDCG@10 (test, MSLR-WEB10k): 0.4439


Обучили CatBoost с `YetiRank` на MSLR-WEB10k (10 000 запросов). В туториале CatBoost на этом датасете типичный NDCG@10 — около 0.48-0.52; ниже видно фактическое значение после обучения.

## 2. Internet Mathematics 2009

### 2.1 Приведение к MS LETOR

Оригинальный формат IMAT2009: `<label> <fid>:<val> ... <qid>` — метка, разрежённые признаки, в комментарии id запроса. Отличия от MS LETOR: нет токена `qid:<N>`, метки могут быть дробными (усреднённые оценки нескольких асессоров)

Переводим в канонический LETOR: ставим `qid:<qid>` сразу после метки, округляем дробную метку до ближайшего целого (LETOR ожидает целые 0-4), оригинальную метку оставляем в комментарии. Заодно считаем базовую статистику

In [4]:
def parse_imat_line(line):
    parts = line.rstrip('\n').split()
    label = float(parts[0])
    tail_idx = parts.index('#') if '#' in parts else len(parts)
    feats = {int(p.split(':')[0]): float(p.split(':')[1]) for p in parts[1:tail_idx]}
    qid = parts[tail_idx+1] if tail_idx+1 < len(parts) else None
    return label, qid, feats

def imat_to_letor(src_path, dst_path):
    n_rows = 0
    labels_orig, qids, fids = [], set(), set()
    with open(src_path) as fin, open(dst_path, 'w') as fout:
        for line in fin:
            label, qid, feats = parse_imat_line(line)
            labels_orig.append(label); qids.add(qid); fids.update(feats)
            label_int = int(round(label))
            feat_str = ' '.join(f'{k}:{feats[k]:.6f}' for k in sorted(feats))
            fout.write(f'{label_int} qid:{qid} {feat_str} # orig_label={label:.4f}\n')
            n_rows += 1
    return {'rows': n_rows, 'queries': len(qids), 'features': len(fids),
            'labels_orig': labels_orig}

train_stat = imat_to_letor(f'{IMAT_DIR}/imat2009_train_new.txt', f'{IMAT_OUT}/train.txt')
test_stat  = imat_to_letor(f'{IMAT_DIR}/imat2009_test_new.txt',  f'{IMAT_OUT}/test.txt')

print(f'train: {train_stat["rows"]:>7} rows, {train_stat["queries"]:>5} queries, features seen: {train_stat["features"]}')
print(f'test : {test_stat["rows"]:>7} rows, {test_stat["queries"]:>5} queries, features seen: {test_stat["features"]}')
print(f'avg docs / query (train): {train_stat["rows"]/train_stat["queries"]:.1f}')
print(f'avg docs / query (test ): {test_stat["rows"]/test_stat["queries"]:.1f}')

train:   77714 rows,  7300 queries, features seen: 245
test :   19576 rows,  1824 queries, features seen: 244
avg docs / query (train): 10.6
avg docs / query (test ): 10.7


In [5]:
# Распределение оригинальных меток и округлённых классов
lab_train = pd.Series(train_stat['labels_orig'])
lab_test  = pd.Series(test_stat['labels_orig'])
print('train: оригинальные метки — статистика')
print(lab_train.describe().round(3).to_string())
print('\nраспределение округлённых классов (train):')
print(lab_train.round().astype(int).value_counts().sort_index().to_string())
print('\nраспределение округлённых классов (test):')
print(lab_test.round().astype(int).value_counts().sort_index().to_string())

train: оригинальные метки — статистика
count    77714.000
mean         1.068
std          0.935
min          0.000
25%          0.000
50%          1.000
75%          2.000
max          4.000

распределение округлённых классов (train):
0    28128
1    20647
2    26138
3     1823
4      978

распределение округлённых классов (test):
0    6975
1    5219
2    6383
3     835
4     164


Краткое описание данных:

* Размер: 77 714 пар запрос-документ в train (7300 запросов) и 19 576 в test (1824 запроса), в среднем 10.7 документа на запрос

* 245 разрежённых признаков, конкретный набор присутствующих признаков меняется от строки к строке 

* Метки — усреднённые оценки асессоров на шкале 0-4 (целые значения составляют подавляющее большинство, остальные — дробные средние). При округлении получаем стандартную шкалу 0-4

### 2.2 Эксперименты: CatBoost + LightGBM

Два ranking-метода поверх LETOR-формата:

1. `CatBoostRanker` с функцией потерь `YetiRank` (listwise) — стандартный бейзлайн CatBoost для ранжирования
2. `LightGBM` с `objective='lambdarank'` (pairwise с lambda-градиентами) — другая распространённая сильная ranking-реализация

Целевая метрика — NDCG. NDCG@10 на тесте

In [6]:
def load_letor(path):
    labels, qids, rows = [], [], []
    all_fids = set()
    with open(path) as f:
        for line in f:
            parts = line.split()
            labels.append(int(parts[0]))
            qids.append(parts[1].split(':')[1])
            feats = {}
            for p in parts[2:]:
                if ':' in p and not p.startswith('#'):
                    k, v = p.split(':')
                    try:
                        k = int(k); v = float(v)
                        feats[k] = v; all_fids.add(k)
                    except ValueError:
                        pass
                elif p == '#':
                    break
            rows.append(feats)
    return labels, qids, rows, all_fids

train_y, train_q, train_r, fids_tr = load_letor(f'{IMAT_OUT}/train.txt')
test_y,  test_q,  test_r,  fids_te = load_letor(f'{IMAT_OUT}/test.txt')
all_fids = sorted(fids_tr | fids_te)
fid_to_col = {f: i for i, f in enumerate(all_fids)}

def to_dense(rows):
    M = np.zeros((len(rows), len(all_fids)), dtype=np.float32)
    for i, fs in enumerate(rows):
        for k, v in fs.items():
            M[i, fid_to_col[k]] = v
    return M

X_train = to_dense(train_r)
X_test  = to_dense(test_r)
y_train = np.asarray(train_y); y_test = np.asarray(test_y)

# Сортируем строки train/test по qid: CatBoost Pool требует, чтобы одинаковые group_id шли подряд
def sort_by_qid(X, y, q):
    order = np.argsort(q, kind='stable')
    return X[order], y[order], np.asarray(q)[order]

X_train, y_train, q_train = sort_by_qid(X_train, y_train, train_q)
X_test,  y_test,  q_test  = sort_by_qid(X_test,  y_test,  test_q)

print('X_train:', X_train.shape, ' X_test:', X_test.shape, ' features:', len(all_fids))

X_train: (77714, 245)  X_test: (19576, 245)  features: 245


In [7]:
# 2.2.a CatBoostRanker + YetiRank
imat_train_pool = Pool(X_train, label=y_train, group_id=q_train)
imat_test_pool  = Pool(X_test,  label=y_test,  group_id=q_test)
model_imat_cb = CatBoostRanker(
    loss_function='YetiRank', iterations=600, learning_rate=0.05, depth=6,
    random_seed=RNG_SEED, eval_metric='NDCG:top=10;type=Exp',
    early_stopping_rounds=50, verbose=100,
)
model_imat_cb.fit(imat_train_pool, eval_set=imat_test_pool, use_best_model=True)
pred_imat_cb = model_imat_cb.predict(imat_test_pool)
ndcg10_cb = ndcg_at_k(y_test, pred_imat_cb, q_test, k=10)
print(f'CatBoost YetiRank, NDCG@10 (test): {ndcg10_cb:.4f}')

0:	test: 0.7610384	best: 0.7610384 (0)	total: 36.6ms	remaining: 21.9s


100:	test: 0.8490689	best: 0.8492257 (97)	total: 3.13s	remaining: 15.5s


200:	test: 0.8538801	best: 0.8538801 (200)	total: 6.23s	remaining: 12.4s


300:	test: 0.8557670	best: 0.8566888 (288)	total: 9.24s	remaining: 9.18s


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8566887669
bestIteration = 288

Shrink model to first 289 iterations.
CatBoost YetiRank, NDCG@10 (test): 0.8567


In [8]:
# 2.2.b LightGBM LambdaRank
def group_sizes(q):
    sizes = []; prev = None; c = 0
    for v in q:
        if prev is None or v == prev:
            c += 1
        else:
            sizes.append(c); c = 1
        prev = v
    sizes.append(c)
    return sizes

g_train = group_sizes(q_train)
g_test  = group_sizes(q_test)

lgb_train = lgb.Dataset(X_train, label=y_train, group=g_train)
lgb_test  = lgb.Dataset(X_test,  label=y_test,  group=g_test, reference=lgb_train)

params = dict(objective='lambdarank', metric='ndcg', ndcg_eval_at=[10],
              learning_rate=0.05, num_leaves=63, min_data_in_leaf=50,
              verbose=-1, seed=RNG_SEED)
model_imat_lgb = lgb.train(
    params, lgb_train, num_boost_round=600, valid_sets=[lgb_test], valid_names=['test'],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)],
)
pred_imat_lgb = model_imat_lgb.predict(X_test, num_iteration=model_imat_lgb.best_iteration)
ndcg10_lgb = ndcg_at_k(y_test, pred_imat_lgb, q_test, k=10)
print(f'LightGBM lambdarank, NDCG@10 (test): {ndcg10_lgb:.4f}')

Training until validation scores don't improve for 50 rounds


[100]	test's ndcg@10: 0.854972


[200]	test's ndcg@10: 0.857671


Early stopping, best iteration is:
[199]	test's ndcg@10: 0.857852
LightGBM lambdarank, NDCG@10 (test): 0.8579


## 3. Learning to rank на wikIR1k

### 3.1 Улучшение BM25

In [9]:
def load_docs(path):
    ids, docs = [], []
    with open(path, 'r', encoding='utf-8') as f:
        next(f)
        for line in f:
            i = line.find(',')
            ids.append(line[:i])
            docs.append(line[i+1:].rstrip('\n').split())
    return ids, docs

doc_ids, docs_orig = load_docs(DOCS_CSV)
print(f'loaded {len(doc_ids):,} docs; avg len = {np.mean([len(d) for d in docs_orig]):.1f} words')

loaded 369,721 docs; avg len = 197.7 words


In [10]:
def load_queries(path):
    df = pd.read_csv(path)
    return {str(i): t for i, t in zip(df['id_left'], df['text_left'])}

def load_qrels_dict(path):
    out = defaultdict(dict)
    with open(path) as f:
        for line in f:
            qid, _, did, rel = line.split()
            out[qid][did] = int(rel)
    return dict(out)

train_queries = load_queries(f'{WIKI_DIR}/training/queries.csv')
test_queries  = load_queries(f'{WIKI_DIR}/test/queries.csv')
train_qrels   = load_qrels_dict(f'{WIKI_DIR}/training/qrels')
test_qrels    = load_qrels_dict(f'{WIKI_DIR}/test/qrels')
test_qrels_trec = list(read_trec_qrels(f'{WIKI_DIR}/test/qrels'))
print(f'train queries: {len(train_queries)}, test queries: {len(test_queries)}')
print(f'train qrels: {sum(len(v) for v in train_qrels.values()):,} pairs')

train queries: 1444, test queries: 100
train qrels: 47,699 pairs


In [11]:
# BM25 + TF-IDF индексы
t0 = time.time()
bm25 = BM25Okapi(docs_orig)
print(f'BM25 built in {time.time()-t0:.1f}s')

t0 = time.time()
tfidf_vec = TfidfVectorizer(analyzer=lambda x: x, preprocessor=lambda x: x,
                            token_pattern=None, lowercase=False, norm='l2')
tfidf_X = tfidf_vec.fit_transform(docs_orig)
print(f'TF-IDF built in {time.time()-t0:.1f}s, vocab={tfidf_X.shape[1]:,}')

doc_id_to_pos = {d: i for i, d in enumerate(doc_ids)}
doc_len = np.asarray(bm25.doc_len, dtype=np.int32)

/Users/smakov/.pyenv/versions/3.11.9/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:532: UserWarning: The parameter 'preprocessor' will not be used since 'analyzer' is callable'
  warnings.warn(


TF-IDF built in 23.6s, vocab=794,568


In [12]:
# Считаем топ-100 BM25 для всех обучающих и тестовых запросов
TOP_K_TRAIN = 100
TOP_K_TEST  = 100

def bm25_topk(q_tokens, k):
    scores = bm25.get_scores(q_tokens)
    idx = np.argpartition(-scores, k-1)[:k]
    idx = idx[np.argsort(-scores[idx])]
    return idx, scores[idx]

def compute_topk_for(queries, k, desc):
    out = {}
    for qid, raw in tqdm(queries.items(), desc=desc):
        idx, sc = bm25_topk(raw.split(), k)
        out[qid] = (idx, sc)
    return out

train_topk = compute_topk_for(train_queries, TOP_K_TRAIN, 'BM25 train top-100')
test_topk  = compute_topk_for(test_queries,  TOP_K_TEST,  'BM25 test  top-100')

In [ ]:
# Признаки:
#   bm25    — базовый скор
#   tfidf   — косинус TF-IDF
#   qlen    — число слов в запросе
#   dlen    — длина документа в словах
#   n_match — число совпавших термов
#   jacc    — расстояние Жаккара
#   sum_idf — сумма IDF совпавших термов
#   win     — минимальное окно в документе, покрывающее как можно больше уникальных термов запроса
avg_dl = bm25.avgdl

def min_window(tokens, qset):
    if not qset:
        return -1.0
    positions = [i for i, t in enumerate(tokens) if t in qset]
    if not positions:
        return -1.0
    present = set(tokens[i] for i in positions)
    need = qset & present
    if len(need) <= 1:
        return float(len(tokens))
    # скользящее окно по позициям
    best = float('inf')
    left = 0
    counts = Counter()
    have = 0
    for right in range(len(positions)):
        tok = tokens[positions[right]]
        if tok in need:
            if counts[tok] == 0:
                have += 1
            counts[tok] += 1
        while have == len(need):
            best = min(best, positions[right] - positions[left] + 1)
            ltok = tokens[positions[left]]
            if ltok in need:
                counts[ltok] -= 1
                if counts[ltok] == 0:
                    have -= 1
            left += 1
    return float(best)

def features_for_pair(q_tokens, q_vec, d_pos):
    tokens = docs_orig[d_pos]
    dset = set(tokens); qset = set(q_tokens)
    matched = qset & dset
    n_match = sum(1 for t in q_tokens if t in dset)
    union = qset | dset
    jacc = len(matched) / len(union) if union else 0.0
    sum_idf = sum(bm25.idf.get(t, 0.0) for t in matched)
    # rank_bm25 не даёт одиночный скор, считаем через формулу вручную
    score = 0.0
    dl = doc_len[d_pos]; denom_norm = 1 - bm25.b + bm25.b * dl / avg_dl
    freqs = bm25.doc_freqs[d_pos]
    for t in q_tokens:
        if t in freqs:
            f = freqs[t]; idf = bm25.idf.get(t, 0.0)
            score += idf * (f * (bm25.k1 + 1) / (f + bm25.k1 * denom_norm))
    cos = float((tfidf_X[d_pos] @ q_vec.T).toarray().ravel()[0]) if q_vec is not None else 0.0
    win = min_window(tokens, qset)
    return [score, cos, len(q_tokens), dl, n_match, jacc, sum_idf, win]

FEATURE_NAMES = ['bm25', 'tfidf_cos', 'qlen', 'dlen', 'n_match', 'jaccard', 'sum_idf', 'min_window']

In [ ]:
rng = np.random.default_rng(RNG_SEED)

train_rows, train_labels, train_group_ids = [], [], []

for qid, raw in tqdm(train_queries.items(), desc='build train features'):
    q_tokens = raw.split()
    if not q_tokens:
        continue
    q_vec = tfidf_vec.transform([q_tokens])
    if q_vec.nnz == 0:
        continue
    pos_ids = [d for d, r in train_qrels.get(qid, {}).items() if r > 0 and d in doc_id_to_pos]
    if not pos_ids:
        continue
    pool_idx = train_topk[qid][0]
    neg_candidates = [doc_ids[i] for i in pool_idx if doc_ids[i] not in train_qrels.get(qid, {})]
    if not neg_candidates:
        continue
    n_neg = min(len(pos_ids), len(neg_candidates))
    neg_ids = rng.choice(neg_candidates, size=n_neg, replace=False).tolist()
    # ограничим positives до такого же количества (баланс)
    if len(pos_ids) > n_neg:
        pos_ids = list(rng.choice(pos_ids, size=n_neg, replace=False))
    for did in pos_ids:
        train_rows.append(features_for_pair(q_tokens, q_vec, doc_id_to_pos[did]))
        train_labels.append(int(train_qrels[qid][did]))
        train_group_ids.append(qid)
    for did in neg_ids:
        train_rows.append(features_for_pair(q_tokens, q_vec, doc_id_to_pos[did]))
        train_labels.append(0)
        train_group_ids.append(qid)

X_wiki_tr = np.asarray(train_rows, dtype=np.float32)
y_wiki_tr = np.asarray(train_labels, dtype=np.int32)
g_wiki_tr = np.asarray(train_group_ids)
print('train matrix:', X_wiki_tr.shape, ' positives:', (y_wiki_tr>0).sum(), ' negatives:', (y_wiki_tr==0).sum())


build train features:   0%|          | 0/1444 [00:00<?, ?it/s]


build train features:   1%|          | 8/1444 [00:00<00:27, 51.54it/s]


build train features:   1%|          | 15/1444 [00:00<00:24, 58.65it/s]


build train features:   1%|▏         | 21/1444 [00:00<00:25, 55.83it/s]


build train features:   2%|▏         | 31/1444 [00:00<00:20, 67.78it/s]

...


build train features:  99%|█████████▉| 1432/1444 [00:22<00:00, 52.31it/s]


build train features: 100%|█████████▉| 1438/1444 [00:22<00:00, 50.52it/s]


build train features: 100%|██████████| 1444/1444 [00:22<00:00, 50.44it/s]


build train features: 100%|██████████| 1444/1444 [00:22<00:00, 63.57it/s]

train matrix: (51736, 8)  positives: 25868  negatives: 25868


In [15]:
# Готовим тестовые признаки: top-100 BM25 на тестовых запросах
test_rows, test_labels, test_group_ids, test_doc_ids = [], [], [], []
for qid, raw in tqdm(test_queries.items(), desc='build test features'):
    q_tokens = raw.split()
    if not q_tokens:
        continue
    q_vec = tfidf_vec.transform([q_tokens])
    for d_pos in test_topk[qid][0]:
        did = doc_ids[d_pos]
        test_rows.append(features_for_pair(q_tokens, q_vec, d_pos))
        test_labels.append(int(test_qrels.get(qid, {}).get(did, 0)))
        test_group_ids.append(qid)
        test_doc_ids.append(did)

X_wiki_te = np.asarray(test_rows, dtype=np.float32)
y_wiki_te = np.asarray(test_labels, dtype=np.int32)
g_wiki_te = np.asarray(test_group_ids)
print('test matrix:', X_wiki_te.shape, ' positives in top-100:', (y_wiki_te>0).sum())


build test features:   0%|          | 0/100 [00:00<?, ?it/s]


build test features:   3%|▎         | 3/100 [00:00<00:04, 24.11it/s]


build test features:   6%|▌         | 6/100 [00:00<00:04, 23.48it/s]


build test features:   9%|▉         | 9/100 [00:00<00:03, 23.97it/s]


build test features:  12%|█▏        | 12/100 [00:00<00:03, 24.39it/s]

...


build test features:  93%|█████████▎| 93/100 [00:03<00:00, 23.46it/s]


build test features:  96%|█████████▌| 96/100 [00:04<00:00, 23.39it/s]


build test features:  99%|█████████▉| 99/100 [00:04<00:00, 23.45it/s]


build test features: 100%|██████████| 100/100 [00:04<00:00, 23.57it/s]

test matrix: (10000, 8)  positives in top-100: 552


In [ ]:
# Сортируем по qid, обучаем CatBoostRanker с YetiRank 
ord_tr = np.argsort(g_wiki_tr, kind='stable')
ord_te = np.argsort(g_wiki_te, kind='stable')
X_tr, y_tr, g_tr = X_wiki_tr[ord_tr], y_wiki_tr[ord_tr], g_wiki_tr[ord_tr]
X_te, y_te, g_te = X_wiki_te[ord_te], y_wiki_te[ord_te], g_wiki_te[ord_te]
doc_ids_te_sorted = np.asarray(test_doc_ids)[ord_te]

wiki_tr_pool = Pool(X_tr, label=y_tr, group_id=g_tr, feature_names=FEATURE_NAMES)
wiki_te_pool = Pool(X_te, label=y_te, group_id=g_te, feature_names=FEATURE_NAMES)

model_wiki = CatBoostRanker(
    loss_function='YetiRank', iterations=400, learning_rate=0.05, depth=6,
    random_seed=RNG_SEED, eval_metric='NDCG:top=20;type=Exp',
    early_stopping_rounds=50, verbose=100,
)
model_wiki.fit(wiki_tr_pool, eval_set=wiki_te_pool, use_best_model=True)

fi = pd.Series(model_wiki.get_feature_importance(type='PredictionValuesChange'),
               index=FEATURE_NAMES).sort_values(ascending=False)
print('\nfeature importance:')
print(fi.round(2).to_string())

0:	test: 0.3409059	best: 0.3409059 (0)	total: 24.2ms	remaining: 9.64s


100:	test: 0.5830006	best: 0.5843540 (88)	total: 1.55s	remaining: 4.59s


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.5843539946
bestIteration = 88

Shrink model to first 89 iterations.

feature importance:
tfidf_cos     34.43
sum_idf       19.24
bm25          16.70
qlen          16.67
min_window     5.99
jaccard        4.73
n_match        1.15
dlen           1.10


In [17]:
# Считаем метрики через ir_measures для реранжированного и бейзлайнового (BM25) вариантов
def run_from_scores(qids, doc_ids_arr, scores):
    out = []
    for qid, did, sc in zip(qids, doc_ids_arr, scores):
        out.append(ir_measures.ScoredDoc(qid, did, float(sc)))
    return out

def run_bm25_baseline():
    out = []
    for qid, (idx, sc) in test_topk.items():
        for d_pos, s in zip(idx, sc):
            out.append(ir_measures.ScoredDoc(qid, doc_ids[d_pos], float(s)))
    return out

pred_scores = model_wiki.predict(wiki_te_pool)
rerank_run  = run_from_scores(g_te, doc_ids_te_sorted, pred_scores)
bm25_run    = run_bm25_baseline()

MEASURES = [P@1, P@10, P@20, MAP, nDCG@20]
m_rerank = ir_measures.calc_aggregate(MEASURES, test_qrels_trec, rerank_run)
m_bm25   = ir_measures.calc_aggregate(MEASURES, test_qrels_trec, bm25_run)

cmp = pd.DataFrame({
    'BM25 (baseline, top-100 pool)': {str(m): m_bm25[m]   for m in MEASURES},
    'CatBoost rerank (3.1)':          {str(m): m_rerank[m] for m in MEASURES},
})
print(cmp.round(4).to_string())

         BM25 (baseline, top-100 pool)  CatBoost rerank (3.1)
P@1                             0.4900                 0.5400
P@10                            0.2120                 0.2060
P@20                            0.1500                 0.1425
AP                              0.1701                 0.1746
nDCG@20                         0.3570                 0.3676


Выводы по 3.1: добавление восьми простых признаков поверх BM25 и переранжирование top-100 через `CatBoostRanker` даёт небольшой, но устойчивый прирост на тесте: nDCG@20 0.367 vs 0.357 у чистого BM25, P@1 0.54 vs 0.49. 

MAP практически не меняется — пулинг top-100 BM25 ограничивает потолок реранжирования. Среди признаков высоко стоят BM25, sum_idf и tfidf-косинус; min_window и jaccard добавляют небольшой вклад

### 3.2 Реконструкция BM25 из компонент

1. TF запросных термов в документе,
2. IDF запросных термов в коллекции,
3. длина документа.

Признаки собираем на тех же обучающих и тестовых парах, что и в 3.1, обучаем `CatBoostRanker` и сравниваем с BM25

In [ ]:
def features_bm25_parts(q_tokens, d_pos):
    tokens = docs_orig[d_pos]; dl = doc_len[d_pos]; dl_norm = dl / avg_dl
    freqs = bm25.doc_freqs[d_pos]
    tf_vals = [freqs.get(t, 0) for t in q_tokens]
    idf_vals = [bm25.idf.get(t, 0.0) for t in q_tokens]
    tf_arr = np.asarray(tf_vals, dtype=np.float32)
    idf_arr = np.asarray(idf_vals, dtype=np.float32)
    # tf*idf покомпонентно
    tfidf_terms = tf_arr * idf_arr
    return [float(tf_arr.sum()), float(tf_arr.mean()),
            float(idf_arr.sum()), float(idf_arr.mean()),
            float(tfidf_terms.sum()),
            float(dl), float(dl_norm),
            int((tf_arr > 0).sum())]

BM25_FEATURE_NAMES = ['sum_tf','avg_tf','sum_idf','avg_idf','sum_tfidf','dlen','dlen_norm','n_match']

def build_parts(rows_idx_groups, queries_map):
    rows, labels, groups, dids = [], [], [], []
    for qid, raw in tqdm(queries_map.items(), desc='parts features'):
        q_tokens = raw.split()
        if not q_tokens: continue
        pool_idx = rows_idx_groups[qid][0]
        for d_pos in pool_idx:
            rows.append(features_bm25_parts(q_tokens, d_pos))
            labels.append(0)  # заполним позже из qrels
            groups.append(qid); dids.append(doc_ids[d_pos])
    return np.asarray(rows, dtype=np.float32), np.asarray(labels), np.asarray(groups), np.asarray(dids)

# Реконструкция — тот же список (qid, docs), что и в 3.1, но с другими признаками
# Пройдёмся заново, чтобы labels были согласованы (positives + negatives из top-100)
parts_tr_rows, parts_tr_labels, parts_tr_groups = [], [], []
for qid, raw in tqdm(train_queries.items(), desc='train parts'):
    q_tokens = raw.split()
    if not q_tokens: continue
    pos_ids = [d for d, r in train_qrels.get(qid, {}).items() if r > 0 and d in doc_id_to_pos]
    if not pos_ids: continue
    pool_idx = train_topk[qid][0]
    neg_candidates = [doc_ids[i] for i in pool_idx if doc_ids[i] not in train_qrels.get(qid, {})]
    if not neg_candidates: continue
    n_neg = min(len(pos_ids), len(neg_candidates))
    neg_ids = rng.choice(neg_candidates, size=n_neg, replace=False).tolist()
    if len(pos_ids) > n_neg:
        pos_ids = list(rng.choice(pos_ids, size=n_neg, replace=False))
    for did in pos_ids:
        parts_tr_rows.append(features_bm25_parts(q_tokens, doc_id_to_pos[did]))
        parts_tr_labels.append(int(train_qrels[qid][did]))
        parts_tr_groups.append(qid)
    for did in neg_ids:
        parts_tr_rows.append(features_bm25_parts(q_tokens, doc_id_to_pos[did]))
        parts_tr_labels.append(0)
        parts_tr_groups.append(qid)

Xp_tr = np.asarray(parts_tr_rows, dtype=np.float32)
yp_tr = np.asarray(parts_tr_labels, dtype=np.int32)
gp_tr = np.asarray(parts_tr_groups)

# Тест: top-100 BM25 на тестовых, признаки-компоненты
parts_te_rows, parts_te_labels, parts_te_groups, parts_te_dids = [], [], [], []
for qid, raw in tqdm(test_queries.items(), desc='test parts'):
    q_tokens = raw.split()
    if not q_tokens: continue
    for d_pos in test_topk[qid][0]:
        parts_te_rows.append(features_bm25_parts(q_tokens, d_pos))
        parts_te_labels.append(int(test_qrels.get(qid, {}).get(doc_ids[d_pos], 0)))
        parts_te_groups.append(qid)
        parts_te_dids.append(doc_ids[d_pos])

Xp_te = np.asarray(parts_te_rows, dtype=np.float32)
yp_te = np.asarray(parts_te_labels, dtype=np.int32)
gp_te = np.asarray(parts_te_groups)
dp_te = np.asarray(parts_te_dids)
print('train:', Xp_tr.shape, ' test:', Xp_te.shape)


train parts:   0%|          | 0/1444 [00:00<?, ?it/s]


train parts:   5%|▌         | 75/1444 [00:00<00:01, 748.34it/s]


train parts:  11%|█         | 155/1444 [00:00<00:01, 778.55it/s]


train parts:  18%|█▊        | 265/1444 [00:00<00:01, 923.90it/s]


train parts:  25%|██▍       | 358/1444 [00:00<00:01, 870.10it/s]

...


test parts:   0%|          | 0/100 [00:00<?, ?it/s]


test parts:  43%|████▎     | 43/100 [00:00<00:00, 419.40it/s]


test parts:  89%|████████▉ | 89/100 [00:00<00:00, 441.01it/s]


test parts: 100%|██████████| 100/100 [00:00<00:00, 426.52it/s]

train: (51786, 8)  test: (10000, 8)


In [19]:
ord_tr = np.argsort(gp_tr, kind='stable'); Xp_tr, yp_tr, gp_tr = Xp_tr[ord_tr], yp_tr[ord_tr], gp_tr[ord_tr]
ord_te = np.argsort(gp_te, kind='stable'); Xp_te, yp_te, gp_te, dp_te = Xp_te[ord_te], yp_te[ord_te], gp_te[ord_te], dp_te[ord_te]

parts_tr_pool = Pool(Xp_tr, label=yp_tr, group_id=gp_tr, feature_names=BM25_FEATURE_NAMES)
parts_te_pool = Pool(Xp_te, label=yp_te, group_id=gp_te, feature_names=BM25_FEATURE_NAMES)

model_parts = CatBoostRanker(
    loss_function='YetiRank', iterations=400, learning_rate=0.05, depth=6,
    random_seed=RNG_SEED, eval_metric='NDCG:top=20;type=Exp',
    early_stopping_rounds=50, verbose=100,
)
model_parts.fit(parts_tr_pool, eval_set=parts_te_pool, use_best_model=True)

pred_parts = model_parts.predict(parts_te_pool)
parts_run  = run_from_scores(gp_te, dp_te, pred_parts)
m_parts = ir_measures.calc_aggregate(MEASURES, test_qrels_trec, parts_run)

cmp32 = pd.DataFrame({
    'BM25 (formula)': {str(m): m_bm25[m]  for m in MEASURES},
    'CatBoost on BM25 parts (3.2)': {str(m): m_parts[m] for m in MEASURES},
})
print(cmp32.round(4).to_string())
print('\nfeature importance:')
print(pd.Series(model_parts.get_feature_importance(type='PredictionValuesChange'),
                index=BM25_FEATURE_NAMES).sort_values(ascending=False).round(2).to_string())

0:	test: 0.2367581	best: 0.2367581 (0)	total: 20.2ms	remaining: 8.06s


100:	test: 0.4707424	best: 0.4707424 (100)	total: 1.72s	remaining: 5.1s


200:	test: 0.4826363	best: 0.4861958 (187)	total: 3.33s	remaining: 3.29s


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.4881282585
bestIteration = 226

Shrink model to first 227 iterations.
         BM25 (formula)  CatBoost on BM25 parts (3.2)
P@1              0.4900                        0.4400
P@10             0.2120                        0.1840
P@20             0.1500                        0.1265
AP               0.1701                        0.1525
nDCG@20          0.3570                        0.3185

feature importance:
sum_tfidf    37.25
avg_idf      27.57
n_match      13.38
sum_idf       7.69
avg_tf        6.85
sum_tf        5.46
dlen_norm     1.03
dlen          0.76


Вывод по 3.2: имея только агрегаты компонент BM25 (sum_tf, sum_idf, длины и т.п.), CatBoost восстанавливает основную часть качества, но не дотягивает до самой формулы (nDCG@20 0.318 vs 0.357) 

Это логично - у модели нет признаков на уровне отдельных пар (term, doc). Чтобы поднять качество до BM25, пришлось бы передавать ranker'у термы (например, top-N по IDF), а не их сумму

## 4. MIRAGE

Каждому из 7 560 вопросов соответствует ровно 5 кандидат-чанков из `doc_pool.json`. В `oracle.json` указан чанк — считаем его единственным релевантным для запроса (бинарная релевантность)

Метрика — `NDCG@5` (вся выдача — пять кандидатов), дополнительно `MRR` и `P@1`

### 4.1 Разбиение по источникам

Делаем split 80/20 по полю `source` — так сохраняется распределение источников в train и test.

In [20]:
with open(f'{MIRAGE_DIR}/dataset.json') as f:
    mirage_dataset = json.load(f)
with open(f'{MIRAGE_DIR}/doc_pool.json') as f:
    mirage_pool = json.load(f)
with open(f'{MIRAGE_DIR}/oracle.json') as f:
    mirage_oracle = json.load(f)

print('queries:', len(mirage_dataset), ' chunks in pool:', len(mirage_pool), ' oracle entries:', len(mirage_oracle))
print('source distribution:', Counter(d['source'] for d in mirage_dataset))

# Собираем чанки по mapped_id -> список из 5 чанков; в каждом чанке есть doc_name и doc_chunk
pool_by_qid = defaultdict(list)
for ch in mirage_pool:
    pool_by_qid[ch['mapped_id']].append(ch)
for qid, chunks in list(pool_by_qid.items())[:1]:
    print(f'пример: qid={qid}, chunks={len(chunks)}, doc_names={[c["doc_name"] for c in chunks]}')

# Бинарная релевантность: чанк считаем supporting, если его doc_chunk совпадает с оракульным
# (совпадение по (mapped_id, doc_chunk) — у oracle есть mapped_id + doc_chunk)
oracle_chunk_by_qid = {k: v['doc_chunk'] for k, v in mirage_oracle.items()}
# Для быстрого сопоставления — хранить первые ~200 символов обычно достаточно, но сверим по целому chunk

supporting_count = 0
for qid, chunks in pool_by_qid.items():
    if qid not in oracle_chunk_by_qid: continue
    oc = oracle_chunk_by_qid[qid]
    matched = any(c['doc_chunk'] == oc for c in chunks)
    if matched: supporting_count += 1
print(f'queries where oracle chunk is among pool chunks: {supporting_count}/{len(pool_by_qid)}')

queries: 7560  chunks in pool: 37800  oracle entries: 7560
source distribution: Counter({'naturalqa': 3578, 'popqa': 3075, 'triviaqa': 584, 'ifqa': 248, 'drop': 75})
пример: qid=ce40d2c4-f403-4736-ace1-7fca9c722aba, chunks=5, doc_names=['John Mayne', 'John Mayne', 'John Dawson Mayne', 'John Dawson Mayne', 'John Dawson Mayne']
queries where oracle chunk is among pool chunks: 7560/7560


In [21]:
# Stratified split по source
random.seed(RNG_SEED)
dataset_by_src = defaultdict(list)
for d in mirage_dataset:
    dataset_by_src[d['source']].append(d)

train_queries_m, test_queries_m = [], []
for src, qs in dataset_by_src.items():
    idx = list(range(len(qs)))
    random.shuffle(idx)
    cut = int(len(idx) * 0.8)
    train_queries_m.extend(qs[i] for i in idx[:cut])
    test_queries_m.extend(qs[i] for i in idx[cut:])
print('train:', len(train_queries_m), ' test:', len(test_queries_m))
print('train sources:', Counter(d['source'] for d in train_queries_m))
print('test sources :', Counter(d['source'] for d in test_queries_m))

train: 6047  test: 1513
train sources: Counter({'naturalqa': 2862, 'popqa': 2460, 'triviaqa': 467, 'ifqa': 198, 'drop': 60})
test sources : Counter({'naturalqa': 716, 'popqa': 615, 'triviaqa': 117, 'ifqa': 50, 'drop': 15})


### 4.2 Признаки

Собираем три группы признаков:

1. Текстовые — документ (title = `doc_name`, body = `doc_chunk`): BM25 title, BM25 body, покрытие термов запроса в title/body, длины, TF-IDF косинус по body

2. Популярность — среднедневные просмотры wiki-страницы за последние 90 дней 

3. Число входящих ссылок на страницу 

In [22]:
# Собираем уникальные doc_names из pool
unique_doc_names = sorted({ch['doc_name'] for ch in mirage_pool})
print(f'уникальных страниц: {len(unique_doc_names)}')

pageviews = {}
inlinks   = {}

уникальных страниц: 16260
кэш pageviews: 16260, кэш inlinks: 16260


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

PV_START, PV_END = '20240101', '20240331'
PV_URL = ('https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/'
          'en.wikipedia/all-access/user/{title}/daily/{start}/{end}')
HEADERS = {'User-Agent': 'hw3-ltr-notebook (educational; contact: student@example.com)'}
SESSION = requests.Session(); SESSION.headers.update(HEADERS)

def fetch_pageviews(title):
    t = urllib.parse.quote(title.replace(' ', '_'), safe='')
    try:
        r = SESSION.get(PV_URL.format(title=t, start=PV_START, end=PV_END), timeout=15)
        if r.status_code == 404: return 0.0
        if r.status_code != 200: return None
        items = r.json().get('items', [])
        return sum(x['views'] for x in items) / max(len(items), 1) if items else 0.0
    except Exception:
        return None

print(f'нужно запросить pageviews для {len(unique_doc_names)} страниц')
with ThreadPoolExecutor(max_workers=24) as ex:
    futures = {ex.submit(fetch_pageviews, t): t for t in unique_doc_names}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='pageviews'):
        t = futures[fut]
        val = fut.result()
        pageviews[t] = 0.0 if val is None else val
pv_arr = np.array(list(pageviews.values()))
print(f'pageviews готовы: mean={pv_arr.mean():.1f}, median={np.median(pv_arr):.1f}, max={pv_arr.max():.0f}')

нужно запросить pageviews для 0 страниц
pageviews готовы: mean=2.3, median=0.0, max=26704


In [ ]:
LC_URL = 'https://linkcount.toolforge.org/api/?project=en.wikipedia.org&page={title}'

def fetch_inlinks(title):
    t = urllib.parse.quote(title, safe='')
    try:
        r = SESSION.get(LC_URL.format(title=t), timeout=15)
        if r.status_code != 200: return None
        return int(r.json().get('wikilinks', {}).get('all', 0))
    except Exception:
        return None

print(f'нужно запросить inlinks для {len(unique_doc_names)} страниц')
with ThreadPoolExecutor(max_workers=16) as ex:
    futures = {ex.submit(fetch_inlinks, t): t for t in unique_doc_names}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='inlinks'):
        t = futures[fut]
        val = fut.result()
        inlinks[t] = 0 if val is None else val
il_arr = np.array(list(inlinks.values()))
print(f'inlinks готовы: mean={il_arr.mean():.1f}, median={np.median(il_arr):.1f}, max={il_arr.max()}')

нужно запросить inlinks для 0 страниц
inlinks готовы: mean=502.3, median=65.0, max=412683


In [ ]:
# Подготовка признаков
def tok(s):
    return re.findall(r"[a-z0-9]+", s.lower())

chunk_titles_tok = [tok(ch['doc_name'])  for ch in mirage_pool]
chunk_bodies_tok = [tok(ch['doc_chunk']) for ch in mirage_pool]

bm25_body  = BM25Okapi(chunk_bodies_tok)
bm25_title = BM25Okapi(chunk_titles_tok)

chunk_idx_of = {}
for i, ch in enumerate(mirage_pool):
    chunk_idx_of.setdefault(ch['mapped_id'], []).append(i)

# Для скорости считаем BM25 по формуле вручную на пять кандидатов запроса
avgdl_body  = bm25_body.avgdl;  dl_body  = np.asarray(bm25_body.doc_len)
avgdl_title = bm25_title.avgdl; dl_title = np.asarray(bm25_title.doc_len)

def bm25_one(q, ch_pos, bm25_obj, dl_arr, avgdl):
    dl = dl_arr[ch_pos]
    denom_norm = 1 - bm25_obj.b + bm25_obj.b * dl / max(avgdl, 1e-9)
    freqs = bm25_obj.doc_freqs[ch_pos]
    s = 0.0
    for t in q:
        if t in freqs:
            f = freqs[t]; idf = bm25_obj.idf.get(t, 0.0)
            s += idf * (f * (bm25_obj.k1 + 1) / (f + bm25_obj.k1 * denom_norm))
    return s

def mirage_features(q_text, ch, ch_pos):
    q = tok(q_text); qset = set(q)
    body = chunk_bodies_tok[ch_pos]; title = chunk_titles_tok[ch_pos]
    tset = set(title); bset = set(body)
    body_score  = bm25_one(q, ch_pos, bm25_body,  dl_body,  avgdl_body)
    title_score = bm25_one(q, ch_pos, bm25_title, dl_title, avgdl_title)
    sum_idf_title = sum(bm25_title.idf.get(w, 0.0) for w in qset & tset)
    sum_idf_body  = sum(bm25_body.idf.get(w, 0.0)  for w in qset & bset)
    n_match_title = sum(1 for w in q if w in tset)
    n_match_body  = sum(1 for w in q if w in bset)
    pv = float(pageviews.get(ch['doc_name'], 0.0))
    il = float(inlinks.get(ch['doc_name'], 0.0))
    return [
        body_score, title_score,
        sum_idf_body, sum_idf_title,
        n_match_body, n_match_title,
        len(q), len(body), len(title),
        math.log1p(pv), math.log1p(il),
    ]

MIRAGE_FEATURES = [
    'bm25_body','bm25_title',
    'sum_idf_body','sum_idf_title',
    'n_match_body','n_match_title',
    'qlen','body_len','title_len',
    'log_pageviews','log_inlinks',
]

In [26]:
def build_mirage_matrix(queries):
    rows, labels, groups, positions = [], [], [], []
    for q in tqdm(queries, desc='mirage feats'):
        qid = q['query_id']
        cand_positions = chunk_idx_of.get(qid, [])
        if not cand_positions: continue
        oracle_chunk = oracle_chunk_by_qid.get(qid)
        for p in cand_positions:
            rows.append(mirage_features(q['query'], mirage_pool[p], p))
            labels.append(1 if mirage_pool[p]['doc_chunk'] == oracle_chunk else 0)
            groups.append(qid); positions.append(p)
    return (np.asarray(rows, dtype=np.float32),
            np.asarray(labels, dtype=np.int32),
            np.asarray(groups),
            np.asarray(positions))

X_m_tr, y_m_tr, g_m_tr, _   = build_mirage_matrix(train_queries_m)
X_m_te, y_m_te, g_m_te, pos_m_te = build_mirage_matrix(test_queries_m)
print('train:', X_m_tr.shape, ' test:', X_m_te.shape, ' pos ratio:', y_m_tr.mean().round(3))


mirage feats:   0%|          | 0/6047 [00:00<?, ?it/s]


mirage feats:  21%|██        | 1277/6047 [00:00<00:00, 12766.69it/s]


mirage feats:  44%|████▍     | 2654/6047 [00:00<00:00, 13349.35it/s]


mirage feats:  66%|██████▌   | 3989/6047 [00:00<00:00, 12418.39it/s]


mirage feats:  87%|████████▋ | 5238/6047 [00:00<00:00, 12372.73it/s]


mirage feats: 100%|██████████| 6047/6047 [00:00<00:00, 12633.53it/s]


mirage feats:   0%|          | 0/1513 [00:00<?, ?it/s]


mirage feats:  92%|█████████▏| 1394/1513 [00:00<00:00, 13939.32it/s]


mirage feats: 100%|██████████| 1513/1513 [00:00<00:00, 13899.62it/s]

train: (30235, 11)  test: (7565, 11)  pos ratio: 0.2


In [27]:
# Сортировка по qid + обучение
ord_tr = np.argsort(g_m_tr, kind='stable'); X_m_tr, y_m_tr, g_m_tr = X_m_tr[ord_tr], y_m_tr[ord_tr], g_m_tr[ord_tr]
ord_te = np.argsort(g_m_te, kind='stable'); X_m_te, y_m_te, g_m_te, pos_m_te = X_m_te[ord_te], y_m_te[ord_te], g_m_te[ord_te], pos_m_te[ord_te]

m_tr_pool = Pool(X_m_tr, label=y_m_tr, group_id=g_m_tr, feature_names=MIRAGE_FEATURES)
m_te_pool = Pool(X_m_te, label=y_m_te, group_id=g_m_te, feature_names=MIRAGE_FEATURES)

model_mirage = CatBoostRanker(
    loss_function='YetiRank', iterations=500, learning_rate=0.05, depth=6,
    random_seed=RNG_SEED, eval_metric='NDCG:top=5;type=Exp',
    early_stopping_rounds=50, verbose=100,
)
model_mirage.fit(m_tr_pool, eval_set=m_te_pool, use_best_model=True)

print('feature importance:')
print(pd.Series(model_mirage.get_feature_importance(type='PredictionValuesChange'),
                index=MIRAGE_FEATURES).sort_values(ascending=False).round(2).to_string())

0:	test: 0.6604037	best: 0.6604037 (0)	total: 23ms	remaining: 11.5s


100:	test: 0.8385519	best: 0.8387500 (99)	total: 572ms	remaining: 2.26s


200:	test: 0.8456653	best: 0.8462702 (194)	total: 1.11s	remaining: 1.65s


Stopped by overfitting detector  (50 iterations wait)

bestTest = 0.8489980734
bestIteration = 223

Shrink model to first 224 iterations.
feature importance:
sum_idf_body     30.86
qlen             18.33
log_inlinks      11.47
bm25_body         9.67
bm25_title        7.00
body_len          6.43
title_len         4.62
sum_idf_title     4.43
n_match_title     4.18
n_match_body      2.90
log_pageviews     0.12


In [ ]:
# Считаем NDCG@5, MRR и P@1 на тесте; сравниваем с бейзлайном (сортировка по bm25_body)
pred = model_mirage.predict(m_te_pool)

def ndcg_mrr_p1(y_true, y_pred, groups):
    idx = 0; ndcg = []; mrr = []; p1 = []
    while idx < len(groups):
        j = idx
        while j < len(groups) and groups[j] == groups[idx]:
            j += 1
        g_y = y_true[idx:j]; g_p = y_pred[idx:j]
        order = np.argsort(-g_p)
        sorted_y = g_y[order]

        gains = 2.0**sorted_y - 1
        discounts = 1.0/np.log2(np.arange(2, len(sorted_y)+2))
        dcg = (gains * discounts).sum()
        ideal = 2.0**np.sort(g_y)[::-1] - 1
        idcg = (ideal * discounts).sum()
        ndcg.append(dcg/idcg if idcg > 0 else 0.0)
        # MRR по первой релевантной
        rel_positions = np.where(sorted_y > 0)[0]
        mrr.append(1.0/(rel_positions[0]+1) if len(rel_positions) else 0.0)
        p1.append(1.0 if sorted_y[0] > 0 else 0.0)
        idx = j
    return np.mean(ndcg), np.mean(mrr), np.mean(p1)

bm25_body_scores = X_m_te[:, MIRAGE_FEATURES.index('bm25_body')]
pv_scores        = X_m_te[:, MIRAGE_FEATURES.index('log_pageviews')]

ndcg_rerank, mrr_rerank, p1_rerank = ndcg_mrr_p1(y_m_te, pred,              g_m_te)
ndcg_bm25,   mrr_bm25,   p1_bm25   = ndcg_mrr_p1(y_m_te, bm25_body_scores,  g_m_te)
ndcg_pv,     mrr_pv,     p1_pv     = ndcg_mrr_p1(y_m_te, pv_scores,         g_m_te)

mirage_results = pd.DataFrame({
    'BM25 (body only)': [ndcg_bm25, mrr_bm25, p1_bm25],
    'Pageviews only':   [ndcg_pv,   mrr_pv,   p1_pv],
    'CatBoost ranker':  [ndcg_rerank, mrr_rerank, p1_rerank],
}, index=['NDCG@5','MRR','P@1']).round(4)
print(mirage_results.to_string())

        BM25 (body only)  Pageviews only  CatBoost ranker
NDCG@5            0.8097          0.9024           0.8492
MRR               0.7461          0.8701           0.7983
P@1               0.5876          0.7951           0.6642


Выводы:

1. CatBoost ranker с полным набором признаков (NDCG@5 = 0.849) лучше BM25 по телу (0.810), но проигрывает однопризнаковому ранжированию по `log_pageviews` (0.902)

2. По feature importance в обученной модели первые места занимают `sum_idf_body`, `qlen` и `log_inlinks`. Сами `log_pageviews` оказываются ниже — модель размазала коррелированные сигналы и недоиспользовала самый сильный один признак

3. Title + body с раздельными BM25/sum_idf даёт стабильный прирост к одиночному BM25 по телу

## 5. Сводные результаты

Все метрики из предыдущих разделов в одной таблице.

In [29]:
summary = pd.DataFrame([
    ['§1 MSLR-WEB10k',        'CatBoost YetiRank',                 f'NDCG@10 = {ndcg10_msrank:.4f}'],
    ['§2 IMAT2009',           'CatBoost YetiRank',                 f'NDCG@10 = {ndcg10_cb:.4f}'],
    ['§2 IMAT2009',           'LightGBM lambdarank',               f'NDCG@10 = {ndcg10_lgb:.4f}'],
    ['§3.1 wikIR1k',          'BM25 baseline (top-100)',           f'nDCG@20 = {m_bm25[nDCG@20]:.4f}, MAP = {m_bm25[MAP]:.4f}'],
    ['§3.1 wikIR1k',          'CatBoost rerank (8 features)',      f'nDCG@20 = {m_rerank[nDCG@20]:.4f}, MAP = {m_rerank[MAP]:.4f}'],
    ['§3.2 wikIR1k',          'CatBoost on BM25 parts only',       f'nDCG@20 = {m_parts[nDCG@20]:.4f}, MAP = {m_parts[MAP]:.4f}'],
    ['§4 MIRAGE',             'BM25 body only',                    f'NDCG@5 = {ndcg_bm25:.4f}, MRR = {mrr_bm25:.4f}'],
    ['§4 MIRAGE',             'log_pageviews only',                f'NDCG@5 = {ndcg_pv:.4f}, MRR = {mrr_pv:.4f}'],
    ['§4 MIRAGE',             'CatBoost (text + popularity)',      f'NDCG@5 = {ndcg_rerank:.4f}, MRR = {mrr_rerank:.4f}'],
], columns=['task', 'model', 'score'])
print(summary.to_string(index=False))

          task                        model                          score
§1 MSLR-WEB10k            CatBoost YetiRank               NDCG@10 = 0.4439
   §2 IMAT2009            CatBoost YetiRank               NDCG@10 = 0.8567
   §2 IMAT2009          LightGBM lambdarank               NDCG@10 = 0.8579
  §3.1 wikIR1k      BM25 baseline (top-100) nDCG@20 = 0.3570, MAP = 0.1701
  §3.1 wikIR1k CatBoost rerank (8 features) nDCG@20 = 0.3676, MAP = 0.1746
  §3.2 wikIR1k  CatBoost on BM25 parts only nDCG@20 = 0.3185, MAP = 0.1525
     §4 MIRAGE               BM25 body only  NDCG@5 = 0.8097, MRR = 0.7461
     §4 MIRAGE           log_pageviews only  NDCG@5 = 0.9024, MRR = 0.8701
     §4 MIRAGE CatBoost (text + popularity)  NDCG@5 = 0.8492, MRR = 0.7983


Выводы

1. На MSLR-WEB10k `CatBoostRanker` с `YetiRank` воспроизводит ожидаемое по туториалу качество (NDCG@10 ≈ 0.44) — пайплайн собран корректно

2. На IMAT2009 CatBoost YetiRank и LightGBM lambdarank дают практически совпадающие NDCG@10 (~0.857)

3. На wikIR1k реранжирование top-100 BM25 через CatBoost с восемью признаками улучшает nDCG@20 0.357 - 0.368 и P@1 0.49 - 0.54. Реконструкция же только из BM25-агрегатов (sum_tf, sum_idf, dlen) даёт 0.318 nDCG@20 — ниже самой формулы

4. На MIRAGE: CatBoost (NDCG@5 = 0.849) обходит BM25-body (0.810), но сильно проигрывает однопризнаковому ранжированию по `log_pageviews` (0.902); distract-чанки берутся со страниц с заведомо меньшей популярностью, поэтому метрика популярности существенна